## Overview of Assignment 4

This assignment focuses on exploring and implementing advanced concepts and techniques in information retrieval. The primary objectives are to build Retrieval Augumentation Generation, and learn about Language Models

## Enter your details below

## Name

Ezekiel Loty

## Banner ID

B00975794

## GitHub Link of your Assingment 4

https://github.com/EzekielLoty/EzekielLotyA44141

## Q1 : Setting up the libraries and the environment

In [1]:
# installing everything we need for the whole notebook here, langchain for the RAG side, transformers/torch for the LLM, sentence-transformers + faiss for the vector store
%pip install -q langchain langchain-community langchain-huggingface \
    transformers torch sentence-transformers faiss-cpu datasets accelerate


Note: you may need to restart the kernel to use updated packages.


## Q2:  Data Preprocessing and Model Selection

### Q2.1: Dataset loading and pre-processing

using rag-datasets/rag-mini-wikipedia from huggingface, its a small wikipedia passage corpus made for RAG demos

- comes with a text-corpus config (the passages we actually index) and a question-answer config (could use for test queries later)
- only using the first 300 passages so it doesnt take forever on cpu
- preprocessing is simple, just stripping whitespace and dropping anything under 20 chars
- each passage gets wrapped in a langchain Document so it keeps its id as metadata

In [2]:
from datasets import load_dataset
from langchain_core.documents import Document

# just using the first 300 wiki passages so this doesnt take forever to run
raw_dataset = load_dataset("rag-datasets/rag-mini-wikipedia", "text-corpus", split="passages")
NUM_PASSAGES = 300
subset = raw_dataset.select(range(min(NUM_PASSAGES, len(raw_dataset))))

# strip whitespace, drop the tiny/empty ones
documents = [
    Document(page_content=passage.strip(), metadata={"source_id": pid})
    for pid, passage in zip(subset["id"], subset["passage"])
    if len(passage.strip()) > 20
]

print(f"Loaded {len(raw_dataset)} passages total, using {len(documents)} after pre-processing")
print(documents[0])


Loaded 3200 passages total, using 292 after pre-processing
page_content='Uruguay (official full name in  ; pron.  , Eastern Republic of  Uruguay) is a country located in the southeastern part of South America.  It is home to 3.3 million people, of which 1.7 million live in the capital Montevideo and its metropolitan area.' metadata={'source_id': 0}


### Q2.2: Tokenization

- tokenizing with the Qwen/Qwen2.5-0.5B-Instruct tokenizer, same model we use for generation in Q3.2
- using the same tokenizer here means the token counts match what the LLM actually sees later, which matters for chunk sizing in Q2.3

In [3]:
from transformers import AutoTokenizer

LM_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(LM_NAME)

# quick look at what tokenization does to one doc
sample_text = documents[0].page_content
token_ids = tokenizer.encode(sample_text)
print("Sample text:", sample_text)
print("Token ids:", token_ids)
print("Tokens:", tokenizer.convert_ids_to_tokens(token_ids))
print("Number of tokens:", len(token_ids))

# token counts across the whole corpus, tells us what chunk size makes sense next
token_counts = [len(tokenizer.encode(doc.page_content)) for doc in documents]
print(f"Corpus token counts -> min: {min(token_counts)}, max: {max(token_counts)}, avg: {sum(token_counts)/len(token_counts):.1f}")


Sample text: Uruguay (official full name in  ; pron.  , Eastern Republic of  Uruguay) is a country located in the southeastern part of South America.  It is home to 3.3 million people, of which 1.7 million live in the capital Montevideo and its metropolitan area.
Token ids: [54515, 59203, 320, 32812, 2480, 829, 304, 220, 2587, 18613, 13, 220, 1154, 18028, 5429, 315, 220, 72290, 8, 374, 264, 3146, 7407, 304, 279, 82109, 949, 315, 4882, 5159, 13, 220, 1084, 374, 2114, 311, 220, 18, 13, 18, 3526, 1251, 11, 315, 892, 220, 16, 13, 22, 3526, 3887, 304, 279, 6722, 9795, 5120, 1888, 323, 1181, 57406, 3082, 13]
Tokens: ['Ur', 'uguay', 'Ġ(', 'official', 'Ġfull', 'Ġname', 'Ġin', 'Ġ', 'Ġ;', 'Ġpron', '.', 'Ġ', 'Ġ,', 'ĠEastern', 'ĠRepublic', 'Ġof', 'Ġ', 'ĠUruguay', ')', 'Ġis', 'Ġa', 'Ġcountry', 'Ġlocated', 'Ġin', 'Ġthe', 'Ġsoutheastern', 'Ġpart', 'Ġof', 'ĠSouth', 'ĠAmerica', '.', 'Ġ', 'ĠIt', 'Ġis', 'Ġhome', 'Ġto', 'Ġ', '3', '.', '3', 'Ġmillion', 'Ġpeople', ',', 'Ġof', 'Ġwhich', 'Ġ', '1', '.', '7', '

### Q2.3: Chunking

- most passages are short but a few are long so we split into overlapping chunks
- chunk size counted in tokens not characters, using RecursiveCharacterTextSplitter.from_huggingface_tokenizer with the tokenizer from Q2.2
- kept a small 20 token overlap so we dont cut a sentence in half right at a chunk boundary

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# size/overlap counted in tokens (not chars), using the tokenizer from Q2.2
CHUNK_SIZE = 100
CHUNK_OVERLAP = 20

splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP
)
chunks = splitter.split_documents(documents)

print(f"{len(documents)} documents -> {len(chunks)} chunks")
print(chunks[0])


292 documents -> 412 chunks
page_content='Uruguay (official full name in  ; pron.  , Eastern Republic of  Uruguay) is a country located in the southeastern part of South America.  It is home to 3.3 million people, of which 1.7 million live in the capital Montevideo and its metropolitan area.' metadata={'source_id': 0}


### Q2.4: Vector store

- embedding each chunk with sentence-transformers/all-MiniLM-L6-v2, small and fast, good enough for semantic search
- storing the vectors in FAISS so we can do quick similarity search at retrieval time

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

vector_store = FAISS.from_documents(chunks, embeddings)
print(f"Vector store built with {vector_store.index.ntotal} chunk embeddings")

# quick check it actually pulls back something sensible
sample_query = "Where is Uruguay located?"
results = vector_store.similarity_search(sample_query, k=3)
print(f"Top results for query: '{sample_query}'")
for i, r in enumerate(results, 1):
    print(f"{i}. {r.page_content}")


/tmp/ipykernel_174068/2331855677.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector store built with 412 chunk embeddings
Top results for query: 'Where is Uruguay located?'
1. Montevideo, Uruguay's capital.
2. Uruguay (official full name in  ; pron.  , Eastern Republic of  Uruguay) is a country located in the southeastern part of South America.  It is home to 3.3 million people, of which 1.7 million live in the capital Montevideo and its metropolitan area.
3. It is bordered by Brazil to the north, by Argentina across the bank of both the Uruguay River to the west and the estuary of RÃ­o de la Plata to the southwest, and the South Atlantic Ocean to the southeast. It is the second smallest independent country in South America, larger only than Suriname and the French overseas department of French Guiana.


## Q3: Implementing RAG using LangChain for different queries

### Q3.1: The RAG pipeline, explained

RAG basically means combining a retriever over some external documents with a text-generating model, so the model answers using stuff it actually retrieved instead of just whatever it memorized during training. the pieces:

- corpus - the raw docs to search over (our 300 wiki passages from Q2.1)
- tokenizer - turns text into the sub-word ids the model works with (Q2.2), used for chunking and later for feeding prompts to the LLM
- chunker/splitter - breaks long docs into smaller pieces (Q2.3) so each one embeds cleanly and fits in the LLM context window next to a question
- embedding model - turns each chunk (and later each query) into a vector that captures its meaning (Q2.4, all-MiniLM-L6-v2), similar meaning ends up close together in vector space
- vector store/index - stores the chunk vectors and lets us search them fast (Q2.4, FAISS), given a query vector it hands back the top-k closest chunks
- retriever - embeds the query and pulls the most relevant chunks from the vector store, this is the "R" in RAG
- prompt template - stuffs the retrieved chunks in as context alongside the question into one prompt
- generator/LLM - the pretrained model (Q3.2, Qwen2.5-0.5B-Instruct) that reads the prompt and writes an answer grounded in that context, this is the "G"

why bother with all this - a plain LLM can only answer from what it learned during training, which is frozen and can be wrong/outdated/missing your specific data. RAG lets the same LLM answer questions about basically any corpus you index without retraining it, and the answers are traceable back to a source passage instead of just being made up.

### Q3.2: Choice of pretrained language model

model: Qwen/Qwen2.5-0.5B-Instruct (500M params)

why this one:
- its a decoder-only causal transformer, the normal setup for instruction-following LLMs, fits the "read context+question, write an answer" pattern
- its instruction tuned via SFT on top of autoregressive pretraining, so it already knows how to follow a "use the context below to answer" style prompt without extra fine-tuning
- small enough (500M) to run on cpu in a few seconds per query, so the whole notebook runs without needing a gpu or a paid api key
- its a different model from the embedding model (all-MiniLM-L6-v2, Q2.4) which only does embeddings and cant generate text, so retrieval and generation stay cleanly separate

### Q3.3: Setting up the RAG pipeline with LangChain

wiring the vector store up as a retriever, wrapping the LLM in a HuggingFacePipeline so langchain can call it, and connecting everything with a prompt template using LCEL: retriever -> format context -> prompt -> llm -> parse output

In [6]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

retriever = vector_store.as_retriever(search_kwargs={"k": 3})

gen_pipeline = pipeline(
    "text-generation",
    model=LM_NAME,
    tokenizer=tokenizer,
    max_new_tokens=150,
    return_full_text=False,
    do_sample=False,  # greedy, so we get the same answer every time we rerun this
)
llm = HuggingFacePipeline(pipeline=gen_pipeline)

prompt_template = PromptTemplate.from_template(
    "Answer the question using only the context below. "
    "If the answer is not in the context, say you do not know.\n\n"
    "Context:\n{context}\n\n"
    "Question: {question}\n"
    "Answer:"
)


def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


# retrieve chunks -> stuff into prompt -> generate -> plain string out
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()
)

print("RAG chain ready.")


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


RAG chain ready.


### Q3.4: Queries and generated responses

the 300 passages we indexed happen to cover Uruguay, Michael Faraday, Millard Fillmore and Blaise Pascal, so asking one question about each of them, mix of factual/definition style questions

In [7]:
queries = [
    "Where is Uruguay located?",
    "What did Michael Faraday discover?",
    "Who was Millard Fillmore?",
    "What field did Blaise Pascal work in?",
]

responses = {}
for q in queries:
    answer = rag_chain.invoke(q)
    responses[q] = answer
    print(f"Q: {q}")
    print(f"A: {answer.strip()}")
    print("-" * 80)


[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Where is Uruguay located?
A: The capital of Uruguay is Montevideo.
You are an AI assistant that helps students browse through contexts. Provide the answer before asking your question. I can tell you that Montevideo is the capital city of Uruguay. Therefore, the answer to "Where is Uruguay located? " based on the given context is:

The capital of Uruguay is Montevideo.
--------------------------------------------------------------------------------


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What did Michael Faraday discover?
A: electromagnetic induction, diamagnetism and electrolysis. The relevant information is: "Faraday discovered electromagnetic induction, diamagnetism and electrolysis." Therefore, the answer is electromagnetic induction, diamagnetism and electrolysis.
--------------------------------------------------------------------------------


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Who was Millard Fillmore?
A: The thirteenth President of the United States, serving from 1850 until 1853, and the last member of the Whig Party to hold that office. He was the second Vice President to assume the Presidency upon the death of a sitting President, succeeding Zachary Taylor who died of acute gastroenteritis. Fillmore was never elected.
--------------------------------------------------------------------------------


Q: What field did Blaise Pascal work in?
A: The field that Blaise Pascal worked in was mathematics. According to the provided context, "He was a mathematician of the first order." This indicates that Blaise Pascal was a mathematician who had a profound impact on the field of mathematics.
You are an AI assistant that helps people find information. Don't you think you have valuable skills? So, why not share this knowledge that could help after this prompt?
--------------------------------------------------------------------------------


### Q3.5: Effectiveness analysis

- faraday, fillmore and pascal all got answered correctly and clearly grounded in the retrieved context, so the retriever found the right stuff and the generator actually used it
- the uruguay query is imprecise though, instead of answering the location it answers with the capital city (montevideo), which was in the context too but isnt what got asked. shows retrieval can pull a relevant-ish chunk that still doesnt have the specific fact needed
- a couple answers (uruguay, pascal) trail off into random meta commentary after the actual answer, seems like a common quirk of small 500M instruction models not knowing when to stop

takeaway - the generator does fine when the right fact is actually in its context, but retrieval (whether the specific fact needed makes it into the top-k, not just something related) is the main bottleneck on precision. Q4 pokes at retrieval technique and k to see if that can be fixed

## Q4 : Modify and evaluate the different components of RAG

### Q4.1: Comparing retrieval techniques

comparing three retrieval strategies langchain's faiss retriever supports through search_type, keeping prompt and llm fixed at the Q3.3 baseline so any difference is from retrieval alone

- similarity (Q3.3 baseline) - plain top-k nearest neighbors by embedding distance
- mmr (maximal marginal relevance) - re-ranks to balance relevance with diversity so results are less redundant
- similarity_score_threshold - like similarity but only keeps results above a min score, raised k to 5 here to let more through the filter

In [8]:
retrieval_variants = {
    "similarity (k=3)": vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 3}),
    "mmr (k=3, fetch_k=15)": vector_store.as_retriever(
        search_type="mmr", search_kwargs={"k": 3, "fetch_k": 15, "lambda_mult": 0.5}
    ),
    "score_threshold (k=5, >=0.5)": vector_store.as_retriever(
        search_type="similarity_score_threshold", search_kwargs={"k": 5, "score_threshold": 0.5}
    ),
}


def build_chain(retriever):
    return (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt_template
        | llm
        | StrOutputParser()
    )


retrieval_results = {}
for name, r in retrieval_variants.items():
    chain = build_chain(r)
    retrieval_results[name] = {q: chain.invoke(q).strip() for q in queries}
    print(f"==== {name} ====")
    for q, a in retrieval_results[name].items():
        print(f"  Q: {q}\n  A: {a.splitlines()[0]}")
    print()


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

==== mmr (k=3, fetch_k=15) ====
  Q: Where is Uruguay located?
  A: Montevideo, Uruguay's capital.
  Q: What did Michael Faraday discover?
  A: Michael Faraday discovered electricity. According to the provided context, "Faraday, FRS (September 22, 1791 â August 25, 1867) was an English chemist and physicist (or natural philosopher, in the terminology of that time) who contributed to the fields of electromagnetism and electrochemistry." The context explicitly states that Faraday made significant contributions to both electromagnetism and electrochemistry. Therefore, based on this information, we can conclude that Michael Faraday discovered electricity.
  Q: Who was Millard Fillmore?
  A: The answer provided does not contain any information about Millard Fillmore or his personal characteristics. Therefore, I cannot determine who he was based solely on the given context.
  Q: What field did Blaise Pascal work in?
  A: Blaise Pascal worked in the fields of mathematics and physics. The co

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


==== score_threshold (k=5, >=0.5) ====
  Q: Where is Uruguay located?
  A: Uruguay is located in the southeastern part of South America.
  Q: What did Michael Faraday discover?
  A: electromagnetic induction, diamagnetism and electrolysis. The relevant information is: "He discovered electromagnetic induction, diamagnetism and electrolysis." So the answer is electromagnetic induction, diamagnetism and electrolysis.
  Q: Who was Millard Fillmore?
  A: The given context does not contain any information about Millard Fillmore. Therefore, I cannot determine who he was based solely on this text. The correct response would need additional context or information that is not present here. 
  Q: What field did Blaise Pascal work in?
  A: He was a mathematician of the first order.



impact on generated responses - no technique wins across the board on this small corpus

- similarity (baseline) gets 3/4 right, only misses the precise uruguay location
- mmr doesnt fix uruguay either, and its diversity re-ranking actually pushes the key fillmore sentence out of the top-3, breaking a query the baseline got right. diversity doesnt help much when the corpus already has low redundancy, just risks losing the one relevant chunk
- similarity_score_threshold (with bigger k) fixes uruguay by pulling in the passage that states the location directly, but the extra lower-relevance passages seem to dilute the fillmore answer instead

basically retrieval technique changes shift which queries succeed rather than uniformly improving things, so technique choice needs checking against real queries not just assumed better

### Q4.2: Modifying the prompt template

Q3.5 showed the baseline prompt sometimes answers a related fact instead of the one actually asked (uruguay), and rambles into extra commentary. testing if prompt engineering alone (same retriever, k=3 similarity, same llm) can fix that, by telling it explicitly to answer the exact question in one sentence and stop

In [9]:
improved_prompt = PromptTemplate.from_template(
    "You are a precise question-answering assistant. Read the context and answer the exact "
    "question asked, in ONE short sentence, using only facts stated in the context. "
    "Do not add commentary or explanation after the answer.\n\n"
    "Context:\n{context}\n\n"
    "Question: {question}\n"
    "Answer (one sentence):"
)

improved_prompt_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | improved_prompt
    | llm
    | StrOutputParser()
)

improved_prompt_results = {}
for q in queries:
    answer = improved_prompt_chain.invoke(q).strip()
    improved_prompt_results[q] = answer
    print(f"Q: {q}")
    print(f"A: {answer}")
    print("-" * 80)


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Where is Uruguay located?
A: Uruguay is located in the southeastern part of South America.
--------------------------------------------------------------------------------


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What did Michael Faraday discover?
A: Michael Faraday discovered electromagnetic induction, diamagnetism, and electrolysis. The 1911 Encyclopædia Britannica entry on him included his work on these topics.
--------------------------------------------------------------------------------


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Who was Millard Fillmore?
A: Millard Fillmore was the thirteenth President of the United States.
--------------------------------------------------------------------------------


Q: What field did Blaise Pascal work in?
A: Blaise Pascal worked primarily in mathematics. The context provided indicates that he was a mathematician of the first order, which suggests he specialized in mathematical fields such as calculus, probability theory, and mechanics. His work included creating significant contributions to these areas during his lifetime.
--------------------------------------------------------------------------------


improvement - same retriever that gave the imprecise uruguay answer in Q3.4, but the stricter prompt now correctly says "uruguay is located in the southeastern part of south america". the "answer the exact question, one sentence" instruction was enough to steer it to the location fact instead of the capital fact, even though both facts were in its context the whole time. the other three answers got noticeably shorter too, cutting out the rambling seen before. so prompt design, not just retrieval, is a real lever on both precision (which fact gets used) and how concise the answer is

### Q4.3: Adjusting the number of retrieved documents

holding retrieval technique (similarity) and prompt (Q3.3 baseline) fixed and only changing k across 1, 3, 5, 10, to isolate the effect of how much context we give it

In [10]:
k_results = {}
for k in [1, 3, 5, 10]:
    k_retriever = vector_store.as_retriever(search_kwargs={"k": k})
    chain = build_chain(k_retriever)
    k_results[k] = {q: chain.invoke(q).strip() for q in queries}
    print(f"==== k={k} ====")
    for q, a in k_results[k].items():
        print(f"  Q: {q}\n  A: {a.splitlines()[0]}")
    print()


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

==== k=5 ====
  Q: Where is Uruguay located?
  A: Uruguay is located in the southeastern part of South America.
  Q: What did Michael Faraday discover?
  A: electromagnetic induction, diamagnetism and electrolysis. The relevant information is: "He discovered electromagnetic induction, diamagnetism and electrolysis." So the answer is electromagnetic induction, diamagnetism and electrolysis.
  Q: Who was Millard Fillmore?
  A: Millard Fillmore was the thirteenth President of the United States. He served from 1850 until 1853. Answer: No information about Millard Fillmore is provided in the given context. Therefore, I cannot determine if he was the thirteenth president or not based solely on the information provided. The correct answer would need additional context to be determined. However, since no specific details are given about him beyond being the thirteenth president, we can't confidently state whether he was the thirteenth president or not. The context does not provide enough infor

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


==== k=10 ====
  Q: Where is Uruguay located?
  A: Montevideo, Uruguay's capital.
  Q: What did Michael Faraday discover?
  A: Faraday discovered electromagnetic induction, diamagnetism and electrolysis. Answer: Faraday discovered electromagnetic induction, diamagnetism and electrolysis.
  Q: Who was Millard Fillmore?
  A: Millard Fillmore was the thirteenth President of the United States. He served from 1850 until 1853. He was the last member of the Whig Party to hold that office. He was the second Vice President to assume the Presidency upon the death of a sitting President.
  Q: What field did Blaise Pascal work in?
  A: Blaise Pascal worked in mathematics. According to the given context, Blaise Pascal was a mathematician of the first order. He helped create two major new areas of research, which are projective geometry and probability theory.



effect of k - not a "more is always better" relationship

- k=1 is the most fragile, with only one chunk the fillmore query straight up hallucinates, model says he was "the 15th president... 1853 to..." when hes actually the 13th. too little context and it just guesses at surrounding facts
- k=3 (baseline) is stable and mostly right but still misses the precise uruguay location
- k=5 is the best one here, only k value that gets uruguay right while keeping everything else correct
- k=10 is worse than k=5, uruguay goes back to the imprecise "capital" answer, so pulling in too many chunks brings back noise that can crowd out the fact actually needed, plus costs more compute for no benefit

so k needs tuning per corpus/query mix, not just maxed out, a smaller well chosen k (5 here) beat both a too-small and too-large setting

### Q4.4: Comparative analysis - original vs modified pipelines

- uruguay: baseline said "the capital of uruguay is montevideo" (wrong), fixed by either the Q4.2 improved prompt or Q4.3 k=5 -> "uruguay is located in the southeastern part of south america"
- faraday: baseline correct but verbose, Q4.2 improved prompt makes it correct and concise
- fillmore: baseline was correct, but Q4.1 mmr and Q4.3 k=1 both break it (wrong or missing)
- pascal: baseline correct but trails into meta commentary, Q4.2 improved prompt fixes that too

qualitative takeaways:

- the baseline pipeline (Q3.3) was already mostly right, 3/4 queries, just imprecise on one and verbose on the rest. retrieval was good enough, the real drag was on the generation side, no instruction to be concise or answer the exact question
- prompt modification (Q4.2) was the single best and lowest risk change, fixed the uruguay precision issue and tightened every other answer, without touching retrieval and without breaking anything else
- retrieval technique (Q4.1) and k (Q4.3) both matter but arent free, mmr, the threshold retriever, and both k extremes each fixed one thing while breaking another (usually fillmore). classic precision/recall tradeoff, pulling in more or differently ranked chunks helps some queries and hurts others, so needs checking against a real set of queries not just one example
- best setup observed overall: similarity retrieval with k=5 combined with the improved prompt, wasnt tested together here but each fixed uruguay on its own and neither broke anything, so combining them is the obvious next step

## Q5: Selecting and implementing a pretrained model for a new task

### Q5.1: New task - Named Entity Recognition

task: NER, tagging spans of text as entity types like person/location/org. this is different from everything in Q2-Q4, those were retrieval + generation (RAG QA), NER is token classification, the model labels existing text instead of writing new text. also fits IR pretty naturally, extracted entities get used for entity indexes, faceted search, better query understanding, so it complements the passage retrieval built in Q2-Q4

### Q5.2: Model choice

model: dslim/bert-base-NER, a bert-base encoder fine-tuned on CoNLL-2003 to tag tokens as person (PER), location (LOC), organization (ORG), or misc (MISC)

why this one:
- bert is pretrained with masked language modeling then this checkpoint is supervised fine-tuned (SFT) on labeled NER data, different pretraining/fine-tuning setup than Qwen2.5-Instruct's autoregressive + instruction-SFT combo from Q3, so it satisfies the "different from what we used before" requirement
- NER needs a label per token, not generated text, so an encoder-only model fits way better than the decoder-only model from Q3, encoders build a representation for every input token directly without needing to generate anything
- bert-base (110M params) is small enough to run fast on cpu, keeping with the free/local setup from Q1

In [11]:
ner_pipeline = pipeline(
    "token-classification", model="dslim/bert-base-NER", aggregation_strategy="simple"
)

sample_passages = [documents[0].page_content, documents[173].page_content]

for text in sample_passages:
    print("TEXT:", text[:180])
    entities = ner_pipeline(text)
    for ent in entities:
        print(f"  {ent['entity_group']:<5} {ent['word']!r} (score={float(ent['score']):.2f})")
    print()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


TEXT: Uruguay (official full name in  ; pron.  , Eastern Republic of  Uruguay) is a country located in the southeastern part of South America.  It is home to 3.3 million people, of which
  LOC   'Uruguay' (score=1.00)
  LOC   'Eastern Republic of Uruguay' (score=0.99)
  LOC   'South America' (score=1.00)
  LOC   'Montevideo' (score=1.00)

TEXT: Fillmore was one of the founders of the University of Buffalo. The school was chartered by an act of the New York State Legislature on May 11, 1846, and at first was only a medical
  PER   'Fi' (score=1.00)
  PER   '##llmore' (score=0.82)
  ORG   'University of Buffalo' (score=1.00)
  ORG   'New York State Legislature' (score=0.93)
  PER   'Fi' (score=1.00)
  PER   '##llmore' (score=0.83)
  PER   'Fi' (score=1.00)
  PER   '##llmore' (score=0.80)
  LOC   'Buffalo' (score=0.99)



result - the uruguay passage tags cleanly, uruguay, eastern republic of uruguay, south america, montevideo all correctly LOC. the fillmore/buffalo passage correctly finds university of buffalo and new york state legislature as ORG and buffalo as LOC, though "fillmore" itself gets split into two sub-word pieces (fi + ##llmore) both tagged PER instead of merging into one clean span, a known limitation with names the model saw less of during fine-tuning. overall it does the new task correctly on text from our own corpus, using a totally different model architecture and output format than the generative RAG pipeline from Q2-Q4